# Prompt Pattern Analysis

This notebook provides in-depth analysis of prompt patterns, including:
- Template structure analysis
- Placeholder patterns
- Prompt complexity metrics
- Semantic clustering
- Quality assessment metrics

In [ ]:
# Import required libraries
import json
import os
import re
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

## 1. Load and Prepare Data

In [ ]:
# Load the dataset
DATA_DIR = Path('../chatgpt-prompt-dataset/data')

def load_all_datasets(data_dir):
    """Load all prompt datasets from the data directory."""
    datasets = {}
    for file_path in data_dir.glob('*.json'):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            datasets[data['category']] = data
    return datasets

# Load all datasets and combine
all_datasets = load_all_datasets(DATA_DIR)

all_prompts = []
for category, data in all_datasets.items():
    for prompt in data.get('prompts', []):
        prompt['category'] = category
        prompt['level'] = data.get('level', 'N/A')
        all_prompts.append(prompt)

df = pd.DataFrame(all_prompts)
print(f"Loaded {len(df)} prompts from {len(all_datasets)} categories")
df.head()

## 2. Placeholder Pattern Analysis

In [ ]:
def extract_placeholders(text):
    """Extract placeholders from prompt text."""
    # Match patterns like [word], [topic], [number], etc.
    pattern = r'\[([^\]]+)\]'
    return re.findall(pattern, text)

# Extract placeholders for all prompts
df['placeholders'] = df['prompt'].apply(extract_placeholders)
df['placeholder_count'] = df['placeholders'].apply(len)

print("Placeholder Statistics:")
print(df['placeholder_count'].describe())
print(f"\nTotal unique placeholder types: {len(set([p for subs in df['placeholders'] for p in subs]))}")

In [ ]:
# Most common placeholder types
all_placeholders = [p for subs in df['placeholders'] for p in subs]
placeholder_counts = Counter(all_placeholders)

print("Top 20 Most Common Placeholders:")
for placeholder, count in placeholder_counts.most_common(20):
    print(f"  [{placeholder}]: {count}")

In [ ]:
# Visualize placeholder distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Placeholder count by category
category_placeholder_avg = df.groupby('category')['placeholder_count'].mean().sort_values(ascending=False)
bars = axes[0].bar(category_placeholder_avg.index, category_placeholder_avg.values, 
                   color=sns.color_palette('husl', len(category_placeholder_avg)))
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Average Placeholders', fontsize=12)
axes[0].set_title('Average Placeholder Count by Category', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Top 15 placeholders
top_placeholders = dict(placeholder_counts.most_common(15))
axes[1].barh(list(top_placeholders.keys())[::-1], list(top_placeholders.values())[::-1],
             color=sns.color_palette('viridis', len(top_placeholders)))
axes[1].set_xlabel('Frequency', fontsize=12)
axes[1].set_title('Top 15 Placeholder Types', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Prompt Structure Analysis

In [ ]:
def analyze_prompt_structure(text):
    """Analyze structural elements of a prompt."""
    analysis = {
        'has_instruction': bool(re.search(r'^(write|create|design|develop|analyze|explain|generate|build|make)', text.lower())),
        'has_question': '?' in text,
        'has_list_request': bool(re.search(r'(list|list of|\d+ items|\d+ points)', text.lower())),
        'has_example_request': bool(re.search(r'(example|sample|instance)', text.lower())),
        'has_format_spec': bool(re.search(r'(include|format|structure|following)', text.lower())),
        'sentence_count': len(re.split(r'[.!?]+', text)) - 1,
        'has_multiple_instructions': text.count(',') > 2 or text.count(';') > 0,
        'has_colon_list': ':' in text and text.count(',') > 1
    }
    return analysis

# Apply structure analysis
structure_df = df['prompt'].apply(analyze_prompt_structure).apply(pd.Series)
df = pd.concat([df, structure_df], axis=1)

print("Structural Features Analysis:")
print(structure_df.sum())

In [ ]:
# Visualize structural features
structural_features = ['has_instruction', 'has_question', 'has_list_request', 
                      'has_example_request', 'has_format_spec', 'has_multiple_instructions']

fig, ax = plt.subplots(figsize=(12, 6))

# Group by category
feature_by_category = df.groupby('category')[structural_features].mean() * 100

feature_by_category.plot(kind='bar', ax=ax, colormap='viridis')
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Percentage of Prompts (%)', fontsize=12)
ax.set_title('Structural Features by Category', fontsize=14, fontweight='bold')
ax.legend(title='Feature', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Complexity Scoring

In [ ]:
def calculate_complexity_score(row):
    """Calculate a complexity score for a prompt."""
    score = 0
    
    # Word count factor (normalized)
    word_count = len(row['prompt'].split())
    score += min(word_count / 10, 10)  # Max 10 points
    
    # Placeholder count factor
    score += row['placeholder_count'] * 2  # 2 points per placeholder
    
    # Structural complexity factors
    if row['has_multiple_instructions']:
        score += 3
    if row['has_format_spec']:
        score += 2
    if row['has_list_request']:
        score += 2
    
    # Level factor
    level_scores = {'beginner': 1, 'intermediate': 3, 'advanced': 5, 'all levels': 2}
    score += level_scores.get(row['level'], 2) * 2
    
    return score

df['complexity_score'] = df.apply(calculate_complexity_score, axis=1)

print("Complexity Score Statistics:")
print(df['complexity_score'].describe())

In [ ]:
# Complexity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by category
categories = df['category'].unique()
data_by_cat = [df[df['category'] == cat]['complexity_score'].values for cat in categories]
bp = axes[0].boxplot(data_by_cat, labels=categories, patch_artist=True)
colors = sns.color_palette('husl', len(categories))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Complexity Score', fontsize=12)
axes[0].set_title('Complexity Score Distribution by Category', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Complexity by level
level_order = ['beginner', 'intermediate', 'advanced', 'all levels']
level_data = [df[df['level'] == level]['complexity_score'].values for level in level_order if level in df['level'].values]
valid_levels = [level for level in level_order if level in df['level'].values]
bp2 = axes[1].boxplot(level_data, labels=valid_levels, patch_artist=True)
level_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'][:len(valid_levels)]
for patch, color in zip(bp2['boxes'], level_colors):
    patch.set_facecolor(color)
axes[1].set_xlabel('Level', fontsize=12)
axes[1].set_ylabel('Complexity Score', fontsize=12)
axes[1].set_title('Complexity Score by Level', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Text Clustering Analysis

In [ ]:
# TF-IDF Vectorization
# Replace placeholders with generic tokens for clustering
def replace_placeholders(text):
    return re.sub(r'\[[^\]]+\]', '[PLACEHOLDER]', text)

df['prompt_clean'] = df['prompt'].apply(replace_placeholders)

# Create TF-IDF matrix
vectorizer = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df['prompt_clean'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

In [ ]:
# Find optimal number of clusters using silhouette score
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(tfidf_matrix)
    score = silhouette_score(tfidf_matrix, labels)
    silhouette_scores.append(score)

# Plot silhouette scores
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, silhouette_scores, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Optimal Cluster Number Analysis', fontsize=14, fontweight='bold')
ax.axvline(x=K_range[np.argmax(silhouette_scores)], color='r', linestyle='--', 
           label=f'Optimal K={K_range[np.argmax(silhouette_scores)]}')
ax.legend()
plt.tight_layout()
plt.show()

optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters: {optimal_k}")

In [ ]:
# Perform clustering with optimal K
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(tfidf_matrix.toarray())
df['pca_1'] = pca_result[:, 0]
df['pca_2'] = pca_result[:, 1]

# Plot clusters
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(df['pca_1'], df['pca_2'], c=df['cluster'], cmap='viridis', 
                    alpha=0.7, s=100, edgecolors='white', linewidth=0.5)
ax.set_xlabel('PCA Component 1', fontsize=12)
ax.set_ylabel('PCA Component 2', fontsize=12)
ax.set_title('Prompt Clustering Visualization (PCA)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze cluster composition
print("Cluster Composition by Category:")
cluster_category = pd.crosstab(df['cluster'], df['category'])
display(cluster_category)

# Get top terms for each cluster
feature_names = vectorizer.get_feature_names_out()
print("\nTop Terms per Cluster:")
for i in range(optimal_k):
    center = kmeans.cluster_centers_[i]
    top_indices = center.argsort()[-5:][::-1]
    top_terms = [feature_names[idx] for idx in top_indices]
    print(f"Cluster {i}: {', '.join(top_terms)}")

## 6. Verb Analysis

In [ ]:
# Extract action verbs from prompts
action_verbs = []
verb_pattern = r'^([A-Z][a-z]+)'

for prompt in df['prompt']:
    match = re.match(verb_pattern, prompt)
    if match:
        action_verbs.append(match.group(1).lower())

verb_counts = Counter(action_verbs)

print("Top 20 Action Verbs Used in Prompts:")
for verb, count in verb_counts.most_common(20):
    print(f"  {verb}: {count}")

In [ ]:
# Visualize action verbs
fig, ax = plt.subplots(figsize=(12, 6))
top_verbs = dict(verb_counts.most_common(15))
bars = ax.bar(top_verbs.keys(), top_verbs.values(), color=sns.color_palette('coolwarm', len(top_verbs)))
ax.set_xlabel('Action Verb', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Top 15 Action Verbs in Prompts', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

for bar, count in zip(bars, top_verbs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
            str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Quality Assessment Metrics

In [ ]:
def assess_prompt_quality(row):
    """Assess various quality metrics for a prompt."""
    prompt = row['prompt']
    
    metrics = {
        'clarity_score': 0,
        'specificity_score': 0,
        'actionability_score': 0
    }
    
    # Clarity: clear instruction at start, reasonable length
    if row['has_instruction']:
        metrics['clarity_score'] += 2
    if 20 <= len(prompt.split()) <= 100:
        metrics['clarity_score'] += 2
    elif len(prompt.split()) > 100:
        metrics['clarity_score'] += 1
        
    # Specificity: placeholders, specific format requests
    metrics['specificity_score'] = min(row['placeholder_count'] * 2, 5)
    if row['has_format_spec']:
        metrics['specificity_score'] += 2
    if row['has_list_request']:
        metrics['specificity_score'] += 1
        
    # Actionability: clear action verbs, measurable outputs
    if row['has_instruction']:
        metrics['actionability_score'] += 2
    if row['has_list_request'] or row['has_example_request']:
        metrics['actionability_score'] += 2
    if row['has_format_spec']:
        metrics['actionability_score'] += 1
        
    return pd.Series(metrics)

# Apply quality assessment
quality_df = df.apply(assess_prompt_quality, axis=1)
df = pd.concat([df, quality_df], axis=1)
df['overall_quality'] = df['clarity_score'] + df['specificity_score'] + df['actionability_score']

print("Quality Metrics Summary:")
print(df[['clarity_score', 'specificity_score', 'actionability_score', 'overall_quality']].describe())

In [ ]:
# Quality radar chart by category
categories = df['category'].unique()
quality_metrics = ['clarity_score', 'specificity_score', 'actionability_score']

# Calculate averages by category
category_quality = df.groupby('category')[quality_metrics].mean()

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(categories))
width = 0.25

for i, metric in enumerate(quality_metrics):
    ax.bar(x + i * width, category_quality[metric], width, label=metric.replace('_', ' ').title())

ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Quality Metrics by Category', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()

## 8. High-Quality Prompt Identification

In [ ]:
# Identify top quality prompts
top_prompts = df.nlargest(10, 'overall_quality')[['title', 'category', 'prompt', 'overall_quality', 'complexity_score']]

print("Top 10 Highest Quality Prompts:")
for i, (_, row) in enumerate(top_prompts.iterrows(), 1):
    print(f"\n{i}. {row['title']} ({row['category']})")
    print(f"   Quality Score: {row['overall_quality']:.1f} | Complexity: {row['complexity_score']:.1f}")
    print(f"   Prompt: {row['prompt'][:100]}...")

## 9. Recommendations for Prompt Improvement

In [ ]:
# Identify prompts with potential for improvement
low_clarity = df[df['clarity_score'] < 3]
low_specificity = df[df['specificity_score'] < 3]
low_actionability = df[df['actionability_score'] < 3]

print("Prompts with Improvement Potential:")
print(f"  Low Clarity (< 3): {len(low_clarity)} prompts")
print(f"  Low Specificity (< 3): {len(low_specificity)} prompts")
print(f"  Low Actionability (< 3): {len(low_actionability)} prompts")

print("\n--- Sample Prompts for Improvement ---")
if len(low_specificity) > 0:
    sample = low_specificity.iloc[0]
    print(f"\nPrompt: {sample['prompt']}")
    print(f"Issue: Low specificity (score: {sample['specificity_score']})")
    print("Suggestion: Add more placeholders or specify output format")

## 10. Export Analysis Results

In [ ]:
# Export analysis results
analysis_results = {
    'placeholder_analysis': {
        'total_unique_placeholders': len(placeholder_counts),
        'avg_placeholders_per_prompt': df['placeholder_count'].mean(),
        'top_placeholders': dict(placeholder_counts.most_common(10))
    },
    'complexity_analysis': {
        'avg_complexity_score': df['complexity_score'].mean(),
        'complexity_by_category': df.groupby('category')['complexity_score'].mean().to_dict(),
        'complexity_by_level': df.groupby('level')['complexity_score'].mean().to_dict()
    },
    'quality_analysis': {
        'avg_overall_quality': df['overall_quality'].mean(),
        'quality_by_category': df.groupby('category')['overall_quality'].mean().to_dict(),
        'high_quality_threshold': df['overall_quality'].quantile(0.75)
    },
    'cluster_analysis': {
        'optimal_clusters': optimal_k,
        'cluster_distribution': df['cluster'].value_counts().to_dict()
    },
    'action_verbs': dict(verb_counts.most_common(15))
}

# Save results
with open('prompt_analysis_results.json', 'w') as f:
    json.dump(analysis_results, f, indent=2)

print("Analysis results saved to prompt_analysis_results.json")
print(json.dumps(analysis_results, indent=2))

## 11. Summary and Insights

### Key Findings:

1. **Placeholder Usage**: Prompts with more placeholders tend to be more reusable but require more user input

2. **Complexity Patterns**: Advanced prompts show significantly higher complexity scores due to:
   - More structural requirements
   - Multiple output specifications
   - Higher placeholder density

3. **Quality Correlation**: Clarity, specificity, and actionability scores vary by category:
   - Coding prompts: High actionability
   - Creative prompts: High specificity
   - Basic prompts: High clarity

4. **Clustering Insights**: Prompts naturally group into distinct clusters based on:
   - Task type (generation vs analysis)
   - Output format requirements
   - Domain specificity

### Recommendations:

- Add more specific placeholders to low-specificity prompts
- Include clear action verbs at the start of prompts
- Specify output format for better actionable prompts
- Balance complexity with clarity for optimal prompt effectiveness